# Once Upon a Mirror, notebook 2: The random mirror

In section 2 of the game, Random drew a card from a hat each morning, and its season at Cinderella's ran against the
Queen's mirror, which knows every probability. Here you write both mirrors in Python, play them on the game's
race season and compare their rates. Then you play them for 1,000 seasons, at Cinderella's and at Rapunzel's.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one only
if you are still stuck. After each task, a check cell tells you whether your code works. It takes about 25
minutes.

In [ ]:
#@title The mirror's kit (run me first) { display-mode: "form" }
# The mirror's kit. Run this cell first; you do not need to read it.
SEASON = 100            # mornings in a season
RACE = 103993           # the seed of the race season: the season of every comparison in the game
NAMES = ['Robot vacuum', 'Rope ladder', 'Leaf blower', 'Memory-foam mattress', 'Satnav']
EMOJI = ['🤖', '🪜', '🍃', '🛏️', '🧭']
_KIT_STREAMS = {"random": 1, "estimate then exploit": 2, "epsilon-greedy": 3, "ucb": 4}

def random_numbers(seed):
    """A stream of random numbers between 0 and 1: next_number = random_numbers(7), then next_number() gives the
    first number, next_number() again the second, and so on. The same seed gives the same numbers as in the game."""
    state = seed & 0xFFFFFFFF

    def next_number():
        nonlocal state
        state = (state + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((state ^ (state >> 15)) * (1 | state)) & 0xFFFFFFFF
        t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296

    return next_number


def strategy_numbers(seed, strategy):
    """A strategy's own stream of random numbers in the season with this seed, as in the game. strategy is one of
    "random", "estimate then exploit", "epsilon-greedy" and "ucb"."""
    return random_numbers(10 * seed + _KIT_STREAMS[strategy])


def season_moods(seed):
    """The customer's mood on each morning of the season with this seed: SEASON numbers between 0 and 1."""
    next_number = random_numbers(seed)
    moods = []
    for _ in range(SEASON):
        moods.append(next_number())
    return moods


def tap(p, mood):
    """1 if the customer taps the product shown, else 0. She taps when her mood is below p, which happens on a
    share p of the mornings. Every mirror of a season meets the same moods, so the comparison is fair."""
    return 1 if mood < p else 0


def _kit_is_board(rows):
    if not isinstance(rows, (list, tuple)) or len(rows) == 0:
        return False
    for row in rows:
        if not isinstance(row, (list, tuple)):
            return False
        for x in row:
            if x not in (0, 1):
                return False
    return True


def _kit_number(x):
    """At most two decimals, as in the game: 0.6, 0.67, 1, 0."""
    text = "%.2f" % x
    return text.rstrip("0").rstrip(".") if "." in text else text


def rate(rows):
    """The rate of a season: taps / mornings over the whole board."""
    if not _kit_is_board(rows):
        print("⏳ rate needs a board: one list of 1s and 0s per product. It got %s." % _kit_short(rows))
        return None
    taps = sum(sum(row) for row in rows)
    mornings = sum(len(row) for row in rows)
    return taps / mornings if mornings else 0


def _kit_short(x):
    text = repr(x)
    return text if len(text) <= 60 else text[:57] + "..."


def show_board(rows, symbols=None, highlight=None, width=30, estimates=True):
    """Print the board as in the game: the estimate p-hat, the product, then one dot per showing, 🟠 a tap and
    ⚪ no tap. highlight: a row (or a list of rows) to mark with ⭐. A row longer than width is cut and its
    remaining dots are counted."""
    if not _kit_is_board(rows):
        print("⏳ No board to show yet: show_board needs one list of 1s and 0s per product. It got %s." % _kit_short(rows))
        return
    symbols = symbols or EMOJI
    marked = highlight if isinstance(highlight, (list, tuple)) else ([] if highlight is None else [highlight])
    if estimates:
        print("    p\u0302")
    for k in range(len(rows)):
        row = rows[k]
        est = ("?" if not row else _kit_number(sum(row) / len(row))) if estimates else ""
        dots = "".join("🟠" if x else "⚪" for x in row[:width])
        if len(row) > width:
            rest = row[width:]
            dots += " ... and %d more (%d 🟠)" % (len(rest), sum(rest))
        name = symbols[k] if k < len(symbols) else "#%d" % k
        print("%5s  %s  %s%s" % (est, name, dots, "  ⭐" if k in marked else ""))

def estimate(row):
    """The estimate of a product: the average of its 1s and 0s (taps / showings); None when it was never shown."""
    if len(row) == 0:
        return None
    return sum(row) / len(row)


def estimates(rows):
    """The estimate of every product on the board, in the order of the rows."""
    return [estimate(row) for row in rows]

In [ ]:
#@title The checks of notebook 2 (run me too) { display-mode: "form" }
# The checks of notebook 2, and play, which runs your mirror calmly. You do not need to read this cell.
# A check prints one short message and never stops the notebook.

# Each test is a season (its probabilities, its seed, its mornings) and what the game's mirror does on it, morning
# by morning: "shown" holds the row shown on each morning and "dots" her answer, 1 a tap and 0 no tap.
_N2_TESTS = {'queen_mirror': [{'about': "the race season at Cinderella's", 'call': 'queen_mirror(cinderella, season_moods(RACE))', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'shown': '0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'dots': '0111010111011101111111011001111101001110000011010011001011110000000111111001111111111100110011100000'}, {'about': "the race season at Rapunzel's", 'call': 'queen_mirror(rapunzel, season_moods(RACE))', 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'shown': '1111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'dots': '1111011111111101111111111101111101101111001011110111011011111001010111111001111111111101111011110010'}, {'about': 'the collection Ready for the ball, from section 3', 'call': 'queen_mirror([0.6, 0.9, 0.6, 0.95, 0.6], season_moods(1))', 'p': [0.6, 0.9, 0.6, 0.95, 0.6], 'names': ['silk gloves', 'tiara', 'evening bag', 'ball gown', 'lipstick'], 'seed': 1, 'mornings': 100, 'shown': '3333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333', 'dots': '1110011110111111111111111111111111101111111111111110111101111111111111111101111111111111111111111111'}, {'about': 'the collection Long hair, from section 3', 'call': 'queen_mirror([0.02, 0.02, 0.95, 0.02, 0.02], season_moods(2))', 'p': [0.02, 0.02, 0.95, 0.02, 0.02], 'names': ['scissors', 'cap', 'ribbon', 'razor', 'haircut voucher'], 'seed': 2, 'mornings': 100, 'shown': '2222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222', 'dots': '1111111111111111110111111111011111110110101111111111101111111111111011111111111111111111111110111111'}, {'about': "another season at Cinderella's", 'call': 'queen_mirror(cinderella, season_moods(3))', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 3, 'mornings': 100, 'shown': '0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000', 'dots': '0111011111110100101010110100101101110000111110111100010010000001110111111101000011100000010001001110'}, {'about': "a short season of 10 mornings at Rapunzel's", 'call': 'queen_mirror(rapunzel, season_moods(4)[:10])', 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 4, 'mornings': 10, 'shown': '1111111111', 'dots': '0111111011'}], 'random_mirror': [{'about': "the race season at Cinderella's", 'call': 'random_mirror(cinderella, season_moods(RACE), strategy_numbers(RACE, "random"))', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'shown': '0323134003341103440130403401012024023122100133144043404042340020022313344230430010420143324301010020', 'dots': '0101000110000000001011010000101101001000000001000001000010110000000010101001011101101000100010100000'}, {'about': "the race season at Rapunzel's", 'call': 'random_mirror(rapunzel, season_moods(RACE), strategy_numbers(RACE, "random"))', 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'shown': '0323134003341103440130403401012024023122100133144043404042340020022313344230430010420143324301010020', 'dots': '0001000100001100000110000001111101001100000001100000000010010000000010101001001111101100000001110000'}, {'about': 'the collection Ready for the ball, from section 3', 'call': 'random_mirror([0.6, 0.9, 0.6, 0.95, 0.6], season_moods(5), strategy_numbers(5, "random"))', 'p': [0.6, 0.9, 0.6, 0.95, 0.6], 'names': ['silk gloves', 'tiara', 'evening bag', 'ball gown', 'lipstick'], 'seed': 5, 'mornings': 100, 'shown': '0020112144143001420434103224014303224023111242240324030433302322104000310111000223241440143014301024', 'dots': '0010110101101111010011111110110110111001111101100101111111011110110110110111010011111101111011101001'}, {'about': 'the collection Long hair, from section 3', 'call': 'random_mirror([0.02, 0.02, 0.95, 0.02, 0.02], season_moods(6), strategy_numbers(6, "random"))', 'p': [0.02, 0.02, 0.95, 0.02, 0.02], 'names': ['scissors', 'cap', 'ribbon', 'razor', 'haircut voucher'], 'seed': 6, 'mornings': 100, 'shown': '2343012212333132422443410043313410423220132031240304441134043113420244033221311124111413400304113021', 'dots': '1100001101000001011000000000000000010110001000100000000000000000010100000110000010000000000000000010'}, {'about': "another season at Cinderella's", 'call': 'random_mirror(cinderella, season_moods(1), strategy_numbers(1, "random"))', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 1, 'mornings': 100, 'shown': '2232420232242330011120222443131034103122404410042212303023422103231214003301144324221204120032202344', 'dots': '0100000000001011110001100001000100001010011001110000011000000011010000100101010100010011001100011100'}, {'about': "a short season of 10 mornings at Cinderella's", 'call': 'random_mirror(cinderella, season_moods(7)[:10], strategy_numbers(7, "random"))', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 7, 'mornings': 10, 'shown': '3033200312', 'dots': '1100011100'}]}
_N2_SUCCESS = {'queen_mirror': "queen_mirror works on 6 seasons, morning by morning as in the game. On Cinderella's race season it shows the robot vacuum every morning: 60 taps in 100 mornings, a rate of 0.6, the game's numbers.", 'random_mirror': "random_mirror works on 6 seasons, card for card as in the game. On Cinderella's race season it gets 31 taps in 100 mornings, a rate of 0.31, the game's numbers."}
_N2_TASKS = {"queen_mirror": "task A", "random_mirror": "task B"}
_N2_INDEX = {"queen_mirror": "best", "random_mirror": "card"}
_N2_GAPS = {}


def _n2_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 70:
        text = text[:67] + "..."
    return text


def _n2_count(n, word):
    """1 dot, 3 dots"""
    if n == 1:
        return "1 " + word
    return "%d %ss" % (n, word)


def _n2_ordinal(n):
    """1st, 2nd, 3rd, 4th, 11th, 21st"""
    if n % 100 in (11, 12, 13):
        return "%dth" % n
    return "%d%s" % (n, {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th"))


def _n2_morning(t):
    """morning 12 (the 13th of the season): Python counts the mornings from 0."""
    return "morning %d (the %s of the season)" % (t, _n2_ordinal(t + 1))


def _n2_product(test, k):
    """the rope ladder (row 1, p = 0.2)"""
    return "the %s (row %d, p = %s)" % (test["names"][k], k, test["p"][k])


def _n2_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    if code in _N2_GAPS:
        return _N2_GAPS[code]
    # A gap such as "dot = ..." stays in the compiled function; a gap on a line of its own is only visible in the
    # source, which Jupyter and Colab keep for every cell
    gap = Ellipsis in code.co_consts
    if not gap:
        import inspect
        try:
            for line in inspect.getsource(fn).splitlines():
                if "..." in line.split("#")[0]:
                    gap = True
        except Exception:
            pass
    _N2_GAPS[code] = gap
    return gap


def _n2_is_board(value, rows=None):
    """True when value is a board: a list of lists of 1s and 0s (and rows lists, when rows is given)."""
    if not isinstance(value, list) or len(value) == 0:
        return False
    if rows is not None and len(value) != rows:
        return False
    for row in value:
        if not isinstance(row, list):
            return False
        for x in row:
            if not isinstance(x, (bool, int, float)) or x not in (0, 1):
                return False
    return True


def _n2_call(fn, inputs):
    """Run fn on the inputs with its prints silenced. Gives back (answer, error, line): error is None when the call
    went through, and line is the line of fn where an error happened, when it can be read. Never raises."""
    import contextlib, io, traceback
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*inputs)
        return answer, None, None
    except Exception as error:
        line = None
        try:
            for frame in traceback.extract_tb(error.__traceback__):
                if frame.filename == fn.__code__.co_filename and frame.line:
                    line = frame.line.strip()
        except Exception:
            line = None
        return None, "%s: %s" % (type(error).__name__, error), line


def _n2_inputs(name, test, upto=None):
    """Fresh inputs for a test, [p, moods] or [p, moods, next_number], and a counter of the numbers drawn."""
    moods = season_moods(test["seed"])[:test["mornings"]]
    if upto is not None:
        moods = moods[:upto]
    inputs = [list(test["p"]), moods]
    drawn = [0]
    if name == "random_mirror":
        hat = strategy_numbers(test["seed"], "random")

        def next_number():
            drawn[0] = drawn[0] + 1
            return hat()
        inputs.append(next_number)
    return inputs, drawn


def _n2_expected(test, upto):
    """The game's board after the first upto mornings of a test."""
    rows = []
    for k in range(len(test["p"])):
        rows.append([])
    for t in range(upto):
        rows[int(test["shown"][t])].append(int(test["dots"][t]))
    return rows


def _n2_first_failure(name, fn):
    """The first test that fn fails, with what it gave, or None when fn passes them all."""
    for test in _N2_TESTS[name]:
        inputs, drawn = _n2_inputs(name, test)
        answer, error, line = _n2_call(fn, inputs)
        expected = _n2_expected(test, test["mornings"])
        same = False
        if error is None:
            try:
                same = _n2_is_board(answer) and answer == expected
            except Exception:
                same = False
        if not same:
            return {"test": test, "answer": answer, "error": error, "line": line, "expected": expected,
                    "drawn": drawn[0]}
    return None


def _n2_sequence(name, fn, test):
    """What fn does on each morning of a test, read from its boards after 1, 2, 3 ... mornings: a list of pairs
    (row shown, her answer). It stops at the first morning that cannot be read that way."""
    seen = []
    before = []
    for k in range(len(test["p"])):
        before.append([])
    for t in range(test["mornings"]):
        inputs, drawn = _n2_inputs(name, test, t + 1)
        answer, error, line = _n2_call(fn, inputs)
        if error is not None or not _n2_is_board(answer, len(test["p"])):
            return seen
        grown = []
        for k in range(len(answer)):
            if answer[k] != before[k]:
                grown.append(k)
        if len(grown) != 1:
            return seen
        k = grown[0]
        if len(answer[k]) != len(before[k]) + 1 or answer[k][:-1] != before[k]:
            return seen
        seen.append((k, int(answer[k][-1])))
        before = answer
    return seen


def _n2_first_difference(test, seen):
    """The first morning on which seen differs from the game's season, or None when no readable morning differs."""
    for t in range(len(seen)):
        if seen[t] != (int(test["shown"][t]), int(test["dots"][t])):
            return t
    return None


def _n2_tap_problem(name, test, t, seen):
    """(what, hint) when fn showed the game's product on morning t and wrote down another answer."""
    moods = season_moods(test["seed"])
    shown, dot = seen[t]
    mood = moods[t]
    p = test["p"][shown]
    right = int(test["dots"][t])
    index = _N2_INDEX[name]
    if right == 1:
        said = "her mood is %.2f, below %s, so she taps: a 1" % (mood, p)
    else:
        said = "her mood is %.2f, above %s, so she does not tap: a 0" % (mood, p)
    what = "on %s your mirror showed %s and %s. Your board has %s" % (_n2_morning(t), _n2_product(test, shown), said,
                                                                      dot)
    # tap(mood, p) in place of tap(p, mood) gives 1 exactly when p < mood, on every morning of the season
    swapped = True
    for m in range(len(seen)):
        if seen[m][1] != (1 if test["p"][seen[m][0]] < moods[m] else 0):
            swapped = False
    if swapped:
        hint = "tap takes the probability first, then the mood: dot = tap(p[%s], moods[morning])." % index
    else:
        hint = ("her answer is dot = tap(p[%s], moods[morning]): the probability of the product shown, and the mood "
                "of the same morning." % index)
    return what, hint


def _n2_rows_problem(name, test, seen, t):
    """(what, hint) when fn wrote the game's answers into another row: the row shown is wrong, the dots are right."""
    right = int(test["shown"][t])
    shown = seen[t][0]
    if len(seen) == test["mornings"]:
        dots_right = True
        for m in range(len(seen)):
            if seen[m][1] != int(test["dots"][m]):
                dots_right = False
        if dots_right:
            index = _N2_INDEX[name]
            return ("on %s her answer went into the row of %s, and the product shown is %s"
                    % (_n2_morning(t), _n2_product(test, shown), _n2_product(test, right)),
                    "the answers are right, and they go into another row. Add each dot to the row of the product "
                    "shown: rows[%s].append(dot)." % index)
    return None


def _n2_explain_queen(fn, failure):
    """(what, hint) for queen_mirror, from what it does morning by morning; None when that cannot be read."""
    test = failure["test"]
    p = test["p"]
    best = int(test["shown"][0])
    seen = _n2_sequence("queen_mirror", fn, test)
    t = _n2_first_difference(test, seen)
    if t is None:
        return None
    shown, dot = seen[t]
    if shown == best:
        return _n2_tap_problem("queen_mirror", test, t, seen)
    problem = _n2_rows_problem("queen_mirror", test, seen, t)
    if problem is not None:
        return problem
    what = ("on %s your mirror showed %s. The Queen's mirror shows %s, the highest probability"
            % (_n2_morning(t), _n2_product(test, shown), _n2_product(test, best)))
    names = getattr(getattr(fn, "__code__", None), "co_names", ())
    if shown == len(p) - 1:
        hint = ("best ends at the last row, so your condition holds for every product. Product i takes the place of "
                "the best only when its probability is higher: if p[i] > p[best].")
    elif p[shown] == min(p):
        hint = "your condition picks the lowest probability. Turn the comparison round: if p[i] > p[best]."
    elif "cinderella" in names or "rapunzel" in names:
        hint = ("use p, the probabilities that the function gets: the check also tries Rapunzel and the collections "
                "of section 3.")
    else:
        hint = ("product i takes the place of the best so far when its probability is higher: if p[i] > p[best]. "
                "After the loop of step 1, best is the row of the highest probability.")
    return what, hint


def _n2_explain_random(fn, failure):
    """(what, hint) for random_mirror: first the numbers drawn from the hat, then what it does morning by morning;
    None when that cannot be read."""
    test = failure["test"]
    p = test["p"]
    mornings = test["mornings"]
    drawn = failure["drawn"]
    names = getattr(getattr(fn, "__code__", None), "co_names", ())
    if drawn == 0:
        if "random" in names or "randint" in names or "choice" in names:
            hint = ("use the hat that the function gets: card = int(next_number() * 5). Python's own random "
                    "numbers give another season than the game's.")
        elif "strategy_numbers" in names or "random_numbers" in names:
            hint = ("use next_number, the hat that the function gets. The check hands it the hat of each season, and a "
                    "hat made inside the function is always the same one.")
        else:
            hint = "draw each morning's card from the hat: card = int(next_number() * 5)."
        return ("your mirror never drew a number from the hat, next_number", hint)
    if drawn == 1 and mornings > 1:
        return ("your mirror drew 1 number from the hat in %d mornings" % mornings,
                "draw a new card every morning: card = int(next_number() * 5) goes inside the loop over the "
                "mornings.")
    if drawn == 2 * mornings:
        return ("your mirror drew 2 numbers from the hat each morning, %d in %d mornings" % (drawn, mornings),
                "call next_number() once each morning, for the card. Every extra call takes a number out of the hat, "
                "and all the cards after it change.")
    if drawn != mornings:
        return ("your mirror drew %d numbers from the hat in %d mornings" % (drawn, mornings),
                "call next_number() exactly once each morning, for the card.")
    seen = _n2_sequence("random_mirror", fn, test)
    t = _n2_first_difference(test, seen)
    if t is None:
        return None
    shown, dot = seen[t]
    right = int(test["shown"][t])
    if shown == right:
        return _n2_tap_problem("random_mirror", test, t, seen)
    problem = _n2_rows_problem("random_mirror", test, seen, t)
    if problem is not None:
        return problem
    what = ("on %s your mirror showed %s. With the same hat, the game's Random shows %s"
            % (_n2_morning(t), _n2_product(test, shown), _n2_product(test, right)))
    cards = []
    for pair in seen:
        cards.append(pair[0])
    expected = []
    for c in test["shown"][:len(seen)]:
        expected.append(int(c))
    best = p.index(max(p))
    if len(seen) > 1 and cards.count(0) == len(cards):
        hint = ("every card is row 0. int(next_number()) is 0 for any number below 1, so multiply first, then take "
                "int: card = int(next_number() * 5).")
    elif len(seen) > 1 and cards.count(best) == len(cards):
        hint = ("your mirror shows %s every morning, the best product. Random draws its card from the hat: "
                "card = int(next_number() * 5)." % _n2_product(test, best))
    elif len(seen) == mornings and max(cards) < len(p) - 1 and max(expected) == len(p) - 1:
        hint = ("your cards never reach row %d. Multiply by 5, the number of products: "
                "int(next_number() * 5) gives every row from 0 to %d." % (len(p) - 1, len(p) - 1))
    else:
        hint = ("each morning's card is int(next_number() * 5), drawn first thing that morning, before her "
                "answer.")
    return what, hint


def _n2_explain_error(name, failure):
    """The hint for a function that stopped with an error."""
    error = failure["error"]
    text = failure["line"] or ""
    test = failure["test"]
    index = _N2_INDEX[name]
    if "positional argument" in error:
        first = {"queen_mirror": "def queen_mirror(p, moods):",
                 "random_mirror": "def random_mirror(p, moods, next_number):"}[name]
        return "keep the first line of the function as it was given: " + first
    if error.startswith("NameError"):
        known = "p, moods, rows, best, morning, dot and tap"
        if name == "random_mirror":
            known = "p, moods, next_number, rows, card, morning, dot and tap"
        return "Python does not know a name in your function: check its spelling. Inside it you can use " + known + "."
    if error.startswith("IndexError"):
        if "p[morning]" in text.replace(" ", ""):
            return ("p has one probability per product, and the season has %d mornings: use p[%s], the probability "
                    "of the product shown." % (test["mornings"], index))
        if name == "random_mirror" and "card" in text:
            return "card must be a row from 0 to %d: int(next_number() * 5) gives exactly that." % (len(test["p"]) - 1)
        if "rows[" in text:
            return ("the board has fewer rows than there are products. It starts with one empty row per product: "
                    "rows.append([]) in the loop over the products.")
        if "moods[" in text:
            return "moods has one mood per morning: moods[morning] is the mood of this morning."
        return ("an index went past the end of a list. Python counts from 0: the rows are 0 to %d, the mornings 0 to "
                "%d." % (len(test["p"]) - 1, test["mornings"] - 1))
    if error.startswith("TypeError"):
        if "indices must be integers" in error and "float" in error:
            if name == "random_mirror":
                return "a row number is a whole number: int(next_number() * 5) turns the number from the hat into one."
            return "best is a row number, a whole number such as 0 or 3: best = i keeps the row of product i."
        if "not supported between" in error:
            return ("tap compares two numbers: the probability of the product shown, p[%s], and the mood of this "
                    "morning, moods[morning]." % index)
        if name == "random_mirror" and ("not callable" in error or "'function'" in error):
            return "next_number is the hat: next_number() with brackets gives the next number."
    if error.startswith("AttributeError") and "append" in error:
        return ("append works on a list. rows[%s] is the row of the product shown, a list, and rows[%s].append(dot) "
                "adds the dot at its end." % (index, index))
    if name == "queen_mirror":
        return "compare your code with the three steps above the cell, then open the hints."
    return "each morning: card = int(next_number() * 5), dot = tap(p[card], moods[morning]), rows[card].append(dot)."


def _n2_shape(name, failure):
    """(what, hint) when fn did not give a board of the right size, or None."""
    test, answer = failure["test"], failure["answer"]
    rows = len(test["p"])
    mornings = test["mornings"]
    index = _N2_INDEX[name]
    if answer is None:
        return ("your function gave no answer (None)", "end the function with return rows, after the loop over the mornings.")
    if not isinstance(answer, list):
        return ("your function gave %s, and a board is a list of rows" % _n2_short(answer),
                "return rows, the board: one list per product.")
    if len(answer) == 0:
        return ("your function gave an empty board, []",
                "the board starts with one empty row per product: rows.append([]) in the loop over the products.")
    for row in answer:
        if not isinstance(row, list):
            return ("your function gave %s, a board with numbers next to its rows" % _n2_short(answer),
                    "add each dot to the row of the product shown: rows[%s].append(dot). rows.append(dot) puts it "
                    "next to the rows." % index)
    if len(answer) != rows:
        return ("your function gave a board of %s for %s" % (_n2_count(len(answer), "row"), _n2_count(rows, "product")),
                "one row per product: the board starts with one empty row for each probability in p.")
    for r in range(rows):
        for x in answer[r]:
            if not isinstance(x, (bool, int, float)) or x not in (0, 1):
                return ("your function gave a board whose row %d holds %s" % (r, _n2_short(x)),
                        "each dot is her answer, 1 or 0: dot = tap(p[%s], moods[morning])." % index)
    dots = 0
    for row in answer:
        dots = dots + len(row)
    if dots == mornings:
        return None
    what = "your function gave a board with %s after %d mornings" % (_n2_count(dots, "dot"), mornings)
    if dots == 0:
        return (what, "each morning adds her answer to a row: rows[%s].append(dot), inside the loop over the "
                      "mornings." % index)
    if dots == 1:
        return (what, "the season stopped after one morning, or the board started again every morning. Start the "
                      "board before the loop over the mornings, and put return rows after that loop, at the left "
                      "edge of the function's body.")
    if dots == rows * mornings:
        return (what, "each morning shows one product, so one row gets the dot. If every row grows together, the rows "
                      "are one and the same list: give each row its own empty list, rows.append([]).")
    return (what, "one dot per morning: append exactly once in each round of the loop over the mornings.")


def _n2_rows_differ(name, failure):
    """(what, hint) from the first row that differs from the game's: the last resort."""
    test, answer, expected = failure["test"], failure["answer"], failure["expected"]
    index = _N2_INDEX[name]
    same_lengths = True
    for k in range(len(expected)):
        if len(answer[k]) != len(expected[k]):
            same_lengths = False
    for k in range(len(expected)):
        if answer[k] != expected[k]:
            what = ("the row of %s has %s and %s, and the game's has %s and %s"
                    % (_n2_product(test, k), _n2_count(len(answer[k]), "dot"), _n2_count(int(sum(answer[k])), "tap"),
                       _n2_count(len(expected[k]), "dot"), _n2_count(sum(expected[k]), "tap")))
            if same_lengths:
                return (what, "every product was shown as often as in the game, and some of her answers differ. Her "
                              "answer is dot = tap(p[%s], moods[morning]): the probability of the product shown, and "
                              "the mood of the same morning." % index)
            if name == "queen_mirror":
                return (what, "each morning: dot = tap(p[best], moods[morning]), then rows[best].append(dot).")
            return (what, "each morning: card = int(next_number() * 5), dot = tap(p[card], moods[morning]), "
                          "rows[card].append(dot).")
    return ("your function gave %s" % _n2_short(answer),
            "compare your code with the steps above the cell, then open the hints.")


def _n2_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    failure = _n2_first_failure(name, fn)
    if failure is None:
        print("✅ " + _N2_SUCCESS[name])
        return
    if _n2_has_gap(fn) or failure["answer"] is Ellipsis or "ellipsis" in str(failure["error"]).lower():
        print("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, run its cell, "
              "then run this check again." % name)
        return
    test = failure["test"]
    head = "%s, %s" % (test["call"], test["about"])
    if failure["error"] is not None:
        where = ""
        if failure["line"]:
            where = " on the line  %s" % failure["line"]
        print("❌ %s: your function stopped with an error%s. %s" % (head, where, failure["error"]))
        print("   💡 Hint: " + _n2_explain_error(name, failure))
        return
    problem = _n2_shape(name, failure)
    if problem is None:
        if name == "queen_mirror":
            problem = _n2_explain_queen(fn, failure)
        else:
            problem = _n2_explain_random(fn, failure)
    if problem is None:
        problem = _n2_rows_differ(name, failure)
    what, hint = problem
    print("❌ %s: %s." % (head, what))
    print("   💡 Hint: " + hint)


def check_queen_mirror(fn):
    """Task A: plays queen_mirror on six seasons (both customers, the two collections of section 3, a short season)
    and compares each board with the game's, morning by morning."""
    try:
        _n2_check("queen_mirror", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the two folded cells at the top again, then this cell." % error)


def check_random_mirror(fn):
    """Task B: plays random_mirror on six seasons with the game's hat of each season and compares each board with
    the game's, card by card."""
    try:
        _n2_check("random_mirror", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the two folded cells at the top again, then this cell." % error)


def play(mirror, *inputs):
    """Plays one season with your mirror and gives its board: play(queen_mirror, cinderella, moods) is
    queen_mirror(cinderella, moods). When the mirror is not finished yet, stops with an error or gives something
    that is not a board, play prints one calm line and gives None, so the cells below still run."""
    try:
        if not callable(mirror):
            print("⏳ play needs your function itself, as in play(queen_mirror, cinderella, moods).")
            return None
        name = getattr(mirror, "__name__", "your mirror")
        if _n2_has_gap(mirror):
            task = ""
            if name in _N2_TASKS:
                task = " (%s)" % _N2_TASKS[name]
            print("⏳ %s is not finished yet%s, so there is no season to show. Finish it, run its cell, then run this "
                  "cell again." % (name, task))
            return None
        copies = []
        for x in inputs:
            if isinstance(x, list):
                copies.append(list(x))
            else:
                copies.append(x)
        answer, error, line = _n2_call(mirror, copies)
        if error is not None:
            print("❌ %s stopped with an error, %s. Its check above says more." % (name, error))
            return None
        rows = None
        if len(copies) > 0 and isinstance(copies[0], list):
            rows = len(copies[0])
        if not _n2_is_board(answer, rows):
            print("❌ %s gave %s, which is not a board. Its check above says more." % (name, _n2_short(answer)))
            return None
        return answer
    except Exception as error:
        print("❌ play could not run (%s). Run the two folded cells at the top again, then this cell." % error)
        return None


print("✅ The checks of notebook 2 are ready.")

## Cinderella and Rapunzel

The game's two customers, with the probability that each of them taps each product. The lists follow the board:
row 0 is the robot vacuum and row 4 the satnav, since Python counts from 0. The numbers are invented for the game. Only the
Queen's mirror may read them; every other mirror sees only the dots on its board. `NAMES` and `EMOJI`, from the
kit, hold the products in the same order.

In [ ]:
# The probability that each customer taps each product, in board order (invented for the game).
# Only the Queen's mirror may read them: every other mirror sees only the dots of its board.
cinderella = [0.6, 0.2, 0.2, 0.4, 0.2]
rapunzel = [0.4, 0.8, 0.2, 0.2, 0.2]

for i in range(len(NAMES)):
    print("row", i, "   Cinderella", cinderella[i], "   Rapunzel", rapunzel[i], "   ", EMOJI[i], NAMES[i])

## A season in Python

A season is 100 mornings. Each morning the customer has a mood, a number between 0 and 1, and she taps the product
shown when her mood is below its probability. Her mood works like the ball drawn from the urn in the game: it falls
below 0.6 on about 6 mornings in 10, so she taps the robot vacuum with probability 0.6.

The kit gives you both parts. `season_moods(seed)` gives her moods for a whole season, and `tap(p, mood)` gives
her answer: 1 for a tap, 0 for no tap. The seed `RACE` is the game's race season, the season of every comparison
in the game.

In [ ]:
# The race season of the game: her mood on each of its mornings
moods = season_moods(RACE)
print("Mornings in the season:", len(moods))

# The second morning (Python counts from 0, so it is moods[1])
mood = moods[1]
print("Her mood on the second morning:", mood)
print("Shown the robot vacuum, p = 0.6. Does she tap?", tap(cinderella[0], mood))
print("Shown the rope ladder, p = 0.2. Does she tap?", tap(cinderella[1], mood))

**Reading the result.** Her mood on the second morning is about 0.34. That is below 0.6, so she would tap the robot vacuum,
and above 0.2, so she would leave the rope ladder alone.

💭 **Think first.** In the game, every mirror of a season meets the same moods: on the second morning of the race
season her mood is 0.34, whichever mirror she looks into. Why is that a fair way to compare two mirrors?

In [ ]:
#@title 💭 One answer (double-click to open) { display-mode: "form" }
# Both mirrors then face the same customer on the same mornings. A morning with a low mood is a lucky
# morning for both of them: she taps any product whose probability is above her mood. So a gap between
# their rates comes from their choices, with less luck mixed in. The game plays every comparison this way.

## 🎯 Task A. The Queen's mirror

The Queen's mirror knows every probability, and every morning it shows the product with the highest one. Write
`queen_mirror(p, moods)`. It plays one season and returns its board: one list per product, in board order, with a
1 for each tap and a 0 for each showing without a tap. A product that is never shown keeps an empty list, `[]`.

Three small steps, each with its gaps:

1. **The best row.** `best` starts at row 0, and the loop looks at every product: product `i` takes its place
   when its probability is higher than `p[best]`.
2. **The board.** One empty row per product. A list grows with `append`: after `rows = []` and
   `rows.append([])`, `rows` is `[[]]`.
3. **The mornings.** Each morning the mirror shows the best product. `tap(p[best], moods[morning])` is her
   answer, 1 or 0, and `append` puts it at the end of that product's row.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# Step 1: p[best] is the highest probability found so far, and p[i] is the probability of product i.
#         Product i takes the place of the best when p[i] > p[best].
# Step 2: the loop runs once per product. In each round, add one empty row: rows.append([]).
# Step 3: dot = tap(p[best], moods[morning]) is her answer this morning.
#         rows[best] is the row of the best product: add dot at its end with append.

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
#     for i in range(len(p)):
#         if p[i] > p[best]:
#             best = i
#
#     for i in range(len(p)):
#         rows.append([])
#
#     for morning in range(len(moods)):
#         dot = tap(p[best], moods[morning])
#         rows[best].append(dot)

In [ ]:
# 🎯 YOUR TURN, task A: the Queen's mirror
def queen_mirror(p, moods):
    # p: the probability of a tap for each product, in board order. The Queen's mirror may read it.
    # moods: her mood on each morning of the season.
    # Returns the board: one list per product, a 1 for each tap and a 0 for each showing without a tap.

    # Step 1: the row with the highest probability
    best = 0
    for i in range(len(p)):
        if ...:                 # 🎯 YOUR TURN: is p[i] higher than p[best]?
            best = i

    # Step 2: the board, one empty row per product
    rows = []
    for i in range(len(p)):
        ...                     # 🎯 YOUR TURN: add an empty row, [], to rows

    # Step 3: every morning, show the best product and write down her answer
    for morning in range(len(moods)):
        dot = ...               # 🎯 YOUR TURN: her answer this morning, 1 or 0 (use tap)
        ...                     # 🎯 YOUR TURN: add dot at the end of the best product's row
    return rows

In [ ]:
check_queen_mirror(queen_mirror)

## The Queen's season

`play(queen_mirror, cinderella, moods)` runs `queen_mirror(cinderella, moods)` and gives its board. While the
function is not finished, it prints one calm line and gives `None`.

💭 **Think first.** The Queen's mirror shows Cinderella the robot vacuum every morning, and she taps it with probability 0.6. About
how many taps do you expect in the 100 mornings of the race season?

In [ ]:
# The Queen's mirror at Cinderella's, on the race season of the game
moods = season_moods(RACE)
queen_rows = play(queen_mirror, cinderella, moods)
show_board(queen_rows)
print("The rate of the Queen's mirror:", rate(queen_rows))

**Reading the result.** The robot vacuum on all 100 mornings and 60 taps: a rate of 0.6, as in the game. The board shows the
first 30 dots of its row and counts the other 70. Its estimate is 0.6 too. The other rows get no dots, so their
estimates stay question marks: the Queen's mirror needs none of them.

## The hat

Random draws a card from a hat each morning: every product has the same chance, 1 in 5, and the card goes back
into the hat. In Python the hat is a stream of random numbers between 0 and 1.
`next_number = strategy_numbers(RACE, "random")` is Random's own hat in the race season, as in the game, and each
call `next_number()` gives the next number. `int(number * 5)` turns a number into a row from 0 to 4: `int`
drops the decimals.

In [ ]:
# Random's hat in the race season: its first 5 numbers, and the card that each one gives
next_number = strategy_numbers(RACE, "random")
for draw in range(5):
    number = next_number()
    card = int(number * 5)
    print(number, "times 5 is", number * 5, "-> row", card, EMOJI[card], NAMES[card])

**Reading the result.** One number, one card. The fifth number times 5 is about 1.03, and `int` drops the
decimals: row 1, the rope ladder. Numbers below 0.2 give row 0, from 0.2 to 0.4 row 1, and so on, so each row gets a
fifth of the numbers. The memory-foam mattress came twice in five mornings: the card goes back into the hat.

## 🎯 Task B. The random mirror

Write `random_mirror(p, moods, next_number)`. The board is started for you, as in step 2 of task A. Each morning:
draw a card from the hat, `int(next_number() * 5)`; then her answer to that product, with `tap`; then the answer
at the end of the card's row.

Random uses `p` only for `tap`, to let the customer answer. The card itself never looks at it. Draw exactly one
number each morning, as the game does: with Random's hat in the race season, `strategy_numbers(RACE, "random")`,
your mirror then plays the game's season card for card.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The card works as in the cell of the hat above: next_number() gives a number between 0 and 1,
# and int(number * 5) turns it into a row from 0 to 4. Call next_number() once each morning.
# Her answer comes from tap, as in task A, with p[card] in the place of p[best].

In [ ]:
#@title 💡 Hint 2 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     for morning in range(len(moods)):
#         card = int(next_number() * 5)
#         dot = tap(p[card], moods[morning])
#         rows[card].append(dot)

In [ ]:
# 🎯 YOUR TURN, task B: the random mirror
def random_mirror(p, moods, next_number):
    # p and moods as in task A. Random uses p only for tap, her answer: the card never looks at it.
    # next_number is the hat: next_number() gives the next number between 0 and 1.
    # Returns the board, as in task A.
    rows = []
    for i in range(len(p)):
        rows.append([])         # one empty row per product

    for morning in range(len(moods)):
        card = ...              # 🎯 YOUR TURN: a card from the hat, a row from 0 to 4
        dot = ...               # 🎯 YOUR TURN: her answer to that product this morning, 1 or 0
        ...                     # 🎯 YOUR TURN: add dot at the end of the card's row
    return rows

In [ ]:
check_random_mirror(random_mirror)

## Random against the Queen's mirror

The same season and the same Cinderella: both mirrors meet the same moods, morning by morning, and Random draws from its
own hat in that season.

In [ ]:
# Random at Cinderella's, on the same race season: the same moods, and Random's own hat in that season
moods = season_moods(RACE)
random_rows = play(random_mirror, cinderella, moods, strategy_numbers(RACE, "random"))
show_board(random_rows)
print("The rate of Random:", rate(random_rows))
print("The rate of the Queen's mirror:", rate(queen_rows))

**Reading the result.** Random spread the 100 mornings over the five products: 30, 15, 14, 21 and 20 showings. It got 31 taps, a rate of
0.31, against 0.6 for the Queen's mirror, as in the game. It showed Cinderella's favourite, the robot vacuum, on 30 mornings, and the
other 70 went to products she taps less often. Its board has an estimate for every product, and Random never looks
at them. They are still rough: the memory-foam mattress, at 0.48, is even a little ahead of the robot vacuum, at 0.47.

## Many seasons

One season is one roll of luck: the next one may go better or worse. `average_rates(p, first_seed, seasons)`
plays both mirrors for many seasons, one season per seed. Each season has its own moods, which both mirrors meet,
and Random its own hat. It adds up all the taps and divides them by all the mornings; `count_taps(rows)` adds up
the 1s of one board.

💭 **Think first.** Each morning Random shows each product with the same chance, 1 in 5, and Cinderella's five
probabilities are 0.6, 0.2, 0.2, 0.4 and 0.2. What rate do you expect from Random over 1,000 seasons? And from the Queen's mirror?

In [ ]:
# Given: the taps of a season, and both mirrors over many seasons
def count_taps(rows):
    # All the 1s on the board, added up
    taps = 0
    for row in rows:
        for dot in row:
            taps = taps + dot
    return taps


def average_rates(p, first_seed, seasons):
    # Plays both mirrors for many seasons, one season per seed, and prints the rate of all those mornings.
    # Each season has its own moods, which both mirrors meet, and Random has its own hat in each season.
    queen_taps = 0
    random_taps = 0
    for season in range(seasons):
        seed = first_seed + season
        moods = season_moods(seed)
        queen_rows = play(queen_mirror, p, moods)
        random_rows = play(random_mirror, p, moods, strategy_numbers(seed, "random"))
        if queen_rows == None:
            return                  # a mirror is not ready yet: play has said why
        if random_rows == None:
            return
        queen_taps = queen_taps + count_taps(queen_rows)
        random_taps = random_taps + count_taps(random_rows)
    mornings = seasons * SEASON
    print("Seasons:", seasons, "   first seed:", first_seed, "   mornings in all:", mornings)
    print("The Queen's mirror:", queen_taps, "taps, a rate of", queen_taps / mornings)
    print("Random:            ", random_taps, "taps, a rate of", random_taps / mornings)

In [ ]:
# 1,000 seasons at Cinderella's: seeds 1 to 1000
average_rates(cinderella, 1, 1000)

**Reading the result.** The Queen's mirror: 0.6002, about 0.6. Random: 0.32002, about the average of her five probabilities,
(0.6 + 0.2 + 0.2 + 0.4 + 0.2) / 5 = 0.32. A card from the hat gives every product one morning in five, so over many seasons Random's rate is the average
of the probabilities. The race season's 0.31 was a typical season for Random.

💭 **Think first.** Rapunzel's probabilities are 0.4, 0.8, 0.2, 0.2 and 0.2. Guess both rates before you run the cell.

In [ ]:
# 1,000 seasons at Rapunzel's: the same seeds
average_rates(rapunzel, 1, 1000)

**Reading the result.** The Queen's mirror: 0.80018, about 0.8, the rope ladder every morning. Random: 0.35896, about the average of
her five probabilities, (0.4 + 0.8 + 0.2 + 0.2 + 0.2) / 5 = 0.36. Random falls further behind at Rapunzel's, about 0.44 below the Queen's mirror against about 0.28
at Cinderella's: her favourite stands far above her other products.

## Back to the game

Random never learns: its board fills up with estimates, and it never reads them. The next mirror does. Section 3
of the game, **Estimate then exploit**, shows each product once and plays Random for n mornings, 20 in the game's
race; then it spends the remaining mornings on the product with the highest estimate. Notebook 3 comes after it.